### Importing libraries 

In [108]:
import torch 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn 
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

In [109]:
# Transformations on the data 

train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    #transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
    ])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [110]:
train_data = datasets.ImageFolder(root=r"D:\Model Knowledge Base\CNN\Seha CNN project\images\train"
                                ,transform=train_transform)

test_data = datasets.ImageFolder(root=r"D:\Model Knowledge Base\CNN\Seha CNN project\images\test"
                                ,transform=test_transform)

In [111]:
print(f"train size: {len(train_data)}\ntest size: {len(test_data)}")

train size: 480
test size: 120


In [112]:
# data loaders 

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=4)
test_loader = DataLoader(test_data, batch_size=32, num_workers=4)

In [113]:
r_images, r_labels = next(iter(train_loader))
s_images, s_labels = next(iter(test_loader))

In [114]:
print(f"train sizes: {r_images.size()} | {r_labels.size()} \ntest sizes: {s_images.size()} | {s_labels.size()}")

train sizes: torch.Size([32, 1, 128, 128]) | torch.Size([32]) 
test sizes: torch.Size([32, 1, 128, 128]) | torch.Size([32])


# Initializing The Network

In [115]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes = 3):
        super(CustomCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3 , padding=1)
        #self.bn3 = nn.BatchNorm2d(64)
        
        self.pool = nn.MaxPool2d(2,2)
        
        self.dropout1 = nn.Dropout(0.5)
        self.dropout2 = nn.Dropout(0.2)
        self.fc1 = nn.Linear(64*16*16, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)
        
    def forward(self, x):
        
        # first convolutional layer 
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        # second convolutional layer 
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        # third convolutional layer 
        x = self.pool(F.relu(self.conv3(x)))
        # flattening 
        x = nn.Flatten()(x)
        #fully connected layers
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)    #################
        x = F.relu(self.fc2(x))
        #x = self.dropout2(x)    #################
        x = self.fc3(x)
        
        return x 

# Training Loop

In [116]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

In [117]:
num_epochs = 15

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        _, predicted = torch.max(outputs,1)
        
        running_loss += loss.item() * labels.size(0)
        
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        epoch_loss = running_loss/total
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        
    train_acc = (correct/total)*100
    cr = classification_report(all_labels, all_preds, target_names=train_data.classes)
    
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | acc: {train_acc:.4f} \n{cr}")
        

Epoch 1/15 | Loss: 1.0435 | acc: 50.8333 
              precision    recall  f1-score   support

       blank       0.60      0.71      0.65       160
      circle       0.45      0.46      0.46       160
       nough       0.45      0.36      0.40       160

    accuracy                           0.51       480
   macro avg       0.50      0.51      0.50       480
weighted avg       0.50      0.51      0.50       480

Epoch 2/15 | Loss: 0.6099 | acc: 67.2917 
              precision    recall  f1-score   support

       blank       0.95      0.99      0.97       160
      circle       0.52      0.49      0.50       160
       nough       0.53      0.53      0.53       160

    accuracy                           0.67       480
   macro avg       0.67      0.67      0.67       480
weighted avg       0.67      0.67      0.67       480

Epoch 3/15 | Loss: 0.5163 | acc: 68.7500 
              precision    recall  f1-score   support

       blank       0.98      1.00      0.99       160
   

In [118]:
model.eval()
test_acc = 0.0
correct = 0 
total = 0
all_labels = []
all_preds = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels ).sum().item()
        total += labels.size(0)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
    test_acc = correct/total
    cr = classification_report(all_labels, all_preds, target_names=test_data.classes)
    print(f"Test acc: {test_acc}\n{cr}")

Test acc: 0.9333333333333333
              precision    recall  f1-score   support

       blank       1.00      1.00      1.00        40
      circle       0.83      1.00      0.91        40
       nough       1.00      0.80      0.89        40

    accuracy                           0.93       120
   macro avg       0.94      0.93      0.93       120
weighted avg       0.94      0.93      0.93       120

